```mermaid
flowchart TB

    %% 输入
    INPUT["Input Feature Map"]

    %% =========================
    %% 分支 1
    %% Average Pooling → 1×1 Conv
    %% =========================
    INPUT --> POOL["Average<br/>Pooling"]
    POOL --> B1["1 × 1 Conv<br/>(24)"]

    %% =========================
    %% 分支 2
    %% 1×1 Conv
    %% =========================
    INPUT --> B2["1 × 1 Conv<br/>(16)"]

    %% =========================
    %% 分支 3
    %% 1×1 Conv → 5×5 Conv
    %% =========================
    INPUT --> B3_1["1 × 1 Conv<br/>(16)"]
    B3_1 --> B3_2["5 × 5 Conv<br/>(24)"]

    %% =========================
    %% 分支 4
    %% 1×1 Conv → 3×3 Conv → 3×3 Conv
    %% =========================
    INPUT --> B4_1["1 × 1 Conv<br/>(16)"]
    B4_1 --> B4_2["3 × 3 Conv<br/>(24)"]
    B4_2 --> B4_3["3 × 3 Conv<br/>(24)"]

    %% =========================
    %% 输出拼接
    %% =========================
    B1 --> OUTPUT["Concatenate"]
    B2 --> OUTPUT
    B3_2 --> OUTPUT
    B4_3 --> OUTPUT

    %% =========================
    %% 样式
    %% =========================
    classDef concat fill:#e8f0df,stroke:#55758c,stroke-width:2px;
    classDef conv fill:#dceaf5,stroke:#55758c,stroke-width:2px;
    classDef conv1x1 fill:#f5edcf,stroke:#55758c,stroke-width:2px;
    classDef pool fill:#f5dfcf,stroke:#55758c,stroke-width:2px;

    class OUTPUT concat;
    class B3_2,B4_2,B4_3 conv;
    class B2,B3_1,B4_1 conv1x1;
    class POOL pool;
```

### 1*1卷积的作用
可以将输入的数据的通道数降低（计算量小，可以降低计算量）

```mermaid
flowchart LR

    A[192 @ 28x28]
    B[5x5 Convolution]
    C[32 @ 28x28]

    A --> B
    B --> C
```
```mermaid
flowchart LR

    A[192 @ 28x28]
    B[1x1 Convolution]
    C[16 @ 28x28]
    D[5x5 Convolution]
    E[32 @ 28x28]

    A --> B
    B --> C
    C --> D
    D --> E
```

### 计算量对比
$$
5^2 \times （28-5+1）^2 \times 192 \times 32
= 88,473,600
$$
$$
1^2 \times 28^2 \times 192 \times 16
+
5^2 \times （28-5+1）^2 \times 16 \times 32
= 9,781,248
$$

In [ ]:
# 第一层 pool层
import torch.nn as nn
import torch
import torch.nn.functional as F

class my_model(torch.nn.Module):
    def __init__(self):
        super(my_model, self).__init__()
        self.branch1_pool = nn.Conv2d(in_channels=3, out_channels=24, kernel_size=1, stride=1, padding=0)
    def forward(self, x):
        x = F.avg_pool2d(x, kernel_size=3, stride=1, padding=1)
        x = self.branch1_pool(x)
        return x

In [ ]:
# 第二层 pool层（1*1卷积）
import torch.nn as nn
import torch
import torch.nn.functional as F

class my_model(torch.nn.Module):
    def __init__(self):
        super(my_model, self).__init__()
        self.branch2_1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=1, stride=1, padding=0)
    def forward(self, x):
        x = self.branch2_1(x)
        return x

In [ ]:
# 第三层 pool层（5*5卷积）
import torch.nn as nn
import torch
import torch.nn.functional as F

class my_model(torch.nn.Module):
    def __init__(self):
        super(my_model, self).__init__()
        self.branch3_1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=1, stride=1, padding=0)
        self.branch3_5 = nn.Conv2d(in_channels=16, out_channels=24, kernel_size=5, stride=1, padding=2)
    def forward(self, x):
        x = self.branch3_1(x)
        x = self.branch3_5(x)
        return x

In [ ]:
# 第四层 pool层（3*3卷积）
import torch.nn as nn
import torch
import torch.nn.functional as F

class my_model(torch.nn.Module):
    def __init__(self):
        super(my_model, self).__init__()
        self.branch4_1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=1, stride=1, padding=0)
        self.branch4_3_1 = nn.Conv2d(in_channels=16, out_channels=24, kernel_size=3, stride=1, padding=1)
        self.branch4_3_2 = nn.Conv2d(in_channels=24, out_channels=24, kernel_size=3, stride=1, padding=1)
    def forward(self, x):
        x = self.branch4_1(x)
        x = self.branch4_3_1(x)
        x = self.branch4_3_2(x)
        return x

In [ ]:
# 将这四层合起来
import torch.nn as nn
import torch
import torch.nn.functional as F

class InceptionA(nn.Module):
    def __init__(self,in_channels):
        super(InceptionA,self).__init__()
        self.branch1_pool = nn.Conv2d(in_channels=in_channels, out_channels=24, kernel_size=1, stride=1, padding=0)

        self.branch2_1 = nn.Conv2d(in_channels=in_channels, out_channels=16, kernel_size=1, stride=1, padding=0)

        self.branch3_1 = nn.Conv2d(in_channels=in_channels, out_channels=16, kernel_size=1, stride=1, padding=0)
        self.branch3_5 = nn.Conv2d(in_channels=16, out_channels=24, kernel_size=5, stride=1, padding=2)

        self.branch4_1 = nn.Conv2d(in_channels=in_channels, out_channels=16, kernel_size=1, stride=1, padding=0)
        self.branch4_3_1 = nn.Conv2d(in_channels=16, out_channels=24, kernel_size=3, stride=1, padding=1)
        self.branch4_3_2 = nn.Conv2d(in_channels=24, out_channels=24, kernel_size=3, stride=1, padding=1)        
    def forward(self, x):
        branch1 = F.avg_pool2d(x, kernel_size=3, stride=1, padding=1)
        branch1 = self.branch1_pool(branch1)

        branch2 = self.branch2_1(x)

        branch3 = self.branch3_1(x)
        branch3 = self.branch3_5(branch3)

        branch4 = self.branch4_1(x)
        branch4 = self.branch4_3_1(branch4)
        branch4 = self.branch4_3_2(branch4)

        outputs = [branch1, branch2, branch3, branch4]
        return torch.cat(outputs, 1)
#构建模型网络
class Net(nn.Module):
    def __init__(self,):
        super(Net,self).__init__()
        self.Conv1 = nn.Conv2d(in_channels=1,out_channels=10,kernel_size=5)
        self.Conv2 = nn.Conv2d(in_channels=88,out_channels=20,kernel_size=5)

        self.incep1 = InceptionA(in_channels=10)
        self.incep2 = InceptionA(in_channels=20)

        self.mp = nn.MaxPool2d(2)
        self.fc = nn.Linear(1408,10)
    def forward(self,x):
        # batch
        in_size = x.size(0)
        x = F.relu(self.mp(self.Conv1(x)))
        x = self.incep1(x)
        x = F.relu(self.mp(self.Conv2(x)))
        x = self.incep2(x)  
        x = x.view(in_size,-1)
        x = self.fc(x)
        return x

